In [14]:
# Cellule 1 — imports
import requests
import pandas as pd
import os
import time

os.makedirs("../data/raw/climat_nasa", exist_ok=True)

In [15]:
# Cellule 2
departements = {
    "Mono":       {"lat":  6.70, "lon":  1.75},
    "Couffo":     {"lat":  7.20, "lon":  1.75},
    "Oueme":      {"lat":  6.47, "lon":  2.50},
    "Plateau":    {"lat":  7.15, "lon":  2.55},
    "Atacora":    {"lat": 10.65, "lon":  1.50},
    "Borgou":     {"lat": 10.20, "lon":  2.90},
    "Littoral":   {"lat":  6.37, "lon":  2.42},
    "Collines":   {"lat":  8.25, "lon":  2.30},
    "Zou":        {"lat":  7.45, "lon":  2.25},
    "Atlantique": {"lat":  6.47, "lon":  2.22},
    "Alibori":    {"lat": 11.45, "lon":  3.15},
    "Donga":      {"lat":  9.60, "lon":  1.85},
}

# Tous les paramètres en une seule requête (max 20 autorisés)
parametres = "PRECTOTCORR,T2M,T2M_MAX,T2M_MIN,RH2M,ALLSKY_SFC_SW_DWN"

print(f"{len(departements)} départements prêts.")

12 départements prêts.


In [16]:
# Cellule 3
def telecharger_nasa_power(nom, lat, lon):
    url = "https://power.larc.nasa.gov/api/temporal/monthly/point"
    params = {
        "parameters": parametres,
        "community":  "AG",
        "longitude":  lon,
        "latitude":   lat,
        "start":      "2003",    # ← juste l'année, pas YYYYMMDD
        "end":        "2022",    # ← idem
        "format":     "CSV",
    }
    print(f"  Téléchargement {nom}...")
    response = requests.get(url, params=params, timeout=60)

    if response.status_code == 200:
        chemin = f"../data/raw/climat_nasa/{nom}.csv"
        with open(chemin, "w", encoding="utf-8") as f:
            f.write(response.text)
        print(f"  OK — {chemin}")
        return True
    else:
        print(f"  ERREUR {response.status_code} : {response.text[:300]}")
        return False

In [17]:
for nom, coords in departements.items():
    print(f"\n--- {nom} ---")
    telecharger_nasa_power(nom, coords["lat"], coords["lon"])
    time.sleep(2)

print("\nTout est téléchargé !")


--- Mono ---
  Téléchargement Mono...
  OK — ../data/raw/climat_nasa/Mono.csv

--- Couffo ---
  Téléchargement Couffo...
  OK — ../data/raw/climat_nasa/Couffo.csv

--- Oueme ---
  Téléchargement Oueme...
  OK — ../data/raw/climat_nasa/Oueme.csv

--- Plateau ---
  Téléchargement Plateau...
  OK — ../data/raw/climat_nasa/Plateau.csv

--- Atacora ---
  Téléchargement Atacora...
  OK — ../data/raw/climat_nasa/Atacora.csv

--- Borgou ---
  Téléchargement Borgou...
  OK — ../data/raw/climat_nasa/Borgou.csv

--- Littoral ---
  Téléchargement Littoral...
  OK — ../data/raw/climat_nasa/Littoral.csv

--- Collines ---
  Téléchargement Collines...
  OK — ../data/raw/climat_nasa/Collines.csv

--- Zou ---
  Téléchargement Zou...
  OK — ../data/raw/climat_nasa/Zou.csv

--- Atlantique ---
  Téléchargement Atlantique...
  OK — ../data/raw/climat_nasa/Atlantique.csv

--- Alibori ---
  Téléchargement Alibori...
  OK — ../data/raw/climat_nasa/Alibori.csv

--- Donga ---
  Téléchargement Donga...
  OK — ..

In [18]:
# Cellule 5 — vérification : lire un fichier pour confirmer que tout va bien
fichier_test = "../data/raw/climat_nasa/Mono.csv"

with open(fichier_test, "r") as f:
    lignes = f.readlines()

# NASA POWER met des métadonnées en haut, les données commencent après -END HEADER-
debut_data = 0
for i, ligne in enumerate(lignes):
    if "-END HEADER-" in ligne:
        debut_data = i + 1
        break

print(f"Données commencent à la ligne {debut_data}")
print("Aperçu :")
for ligne in lignes[debut_data:debut_data+5]:
    print(ligne.strip())

Données commencent à la ligne 27
Aperçu :

PARAMETER,YEAR,JAN,FEB,MAR,APR,MAY,JUN,JUL,AUG,SEP,OCT,NOV,DEC,ANN
ALLSKY_SFC_SW_DWN,2003,17.42,18.9,17.64,18.7,19.01,15.46,17.57,16.61,17.04,18.91,18.8,18.22,17.85
ALLSKY_SFC_SW_DWN,2004,17.68,17.21,17.6,17.98,16.35,15.24,16.16,15.6,16.66,18.46,19.11,18.61,17.22
ALLSKY_SFC_SW_DWN,2005,17.72,17.33,18.93,17.79,18.57,15.34,14.5,15.46,16.92,20.23,19.31,18.59,17.56


In [19]:
# Lecture et fusion de tous les fichiers NASA POWER

from io import StringIO
import os

def lire_nasa_power(nom, chemin):
    with open(chemin, "r", encoding="utf-8") as f:
        lignes = f.readlines()
    
    debut = 0
    for i, ligne in enumerate(lignes):
        if "-END HEADER-" in ligne:
            debut = i + 1
            break
    
    contenu = "".join(lignes[debut:])
    df = pd.read_csv(StringIO(contenu))
    
    mois = ["JAN","FEB","MAR","APR","MAY","JUN","JUL","AUG","SEP","OCT","NOV","DEC"]
    df_mois = df[["PARAMETER","YEAR"] + mois].copy()
    
    df_long = df_mois.melt(
        id_vars=["PARAMETER","YEAR"],
        value_vars=mois,
        var_name="MOIS",
        value_name="VALEUR"
    )
    
    df_pivot = df_long.pivot_table(
        index=["YEAR","MOIS"],
        columns="PARAMETER",
        values="VALEUR"
    ).reset_index()
    
    df_pivot.columns.name = None
    df_pivot["departement"] = nom
    df_pivot["annee"] = df_pivot["YEAR"]
    
    return df_pivot

dossier = "../data/raw/climat_nasa"
frames = []

for nom in departements.keys():
    chemin = f"{dossier}/{nom}.csv"
    if os.path.exists(chemin):
        df = lire_nasa_power(nom, chemin)
        frames.append(df)
        print(f"OK : {nom} — {len(df)} lignes")
    else:
        print(f"MANQUANT : {nom}")

climat_mensuel = pd.concat(frames, ignore_index=True)
print(f"\nDataset climatique mensuel : {climat_mensuel.shape}")
print(climat_mensuel.head())

OK : Mono — 240 lignes
OK : Couffo — 240 lignes
OK : Oueme — 240 lignes
OK : Plateau — 240 lignes
OK : Atacora — 240 lignes
OK : Borgou — 240 lignes
OK : Littoral — 240 lignes
OK : Collines — 240 lignes
OK : Zou — 240 lignes
OK : Atlantique — 240 lignes
OK : Alibori — 240 lignes
OK : Donga — 240 lignes

Dataset climatique mensuel : (2880, 10)
   YEAR MOIS  ALLSKY_SFC_SW_DWN  PRECTOTCORR   RH2M    T2M  T2M_MAX  T2M_MIN  \
0  2003  APR              18.70         4.60  83.44  27.81    33.50    24.15   
1  2003  AUG              16.61         2.52  88.01  24.60    28.78    21.26   
2  2003  DEC              18.22         0.67  79.27  26.24    30.83    19.16   
3  2003  FEB              18.90         0.64  78.77  28.56    35.27    24.89   
4  2003  JAN              17.42         0.74  76.54  27.23    33.39    18.63   

  departement  annee  
0        Mono   2003  
1        Mono   2003  
2        Mono   2003  
3        Mono   2003  
4        Mono   2003  


In [20]:
# Agréger par année et département — tous les mois conservés

mois_map = {
    "JAN": 1, "FEB": 2, "MAR": 3, "APR": 4,
    "MAY": 5, "JUN": 6, "JUL": 7, "AUG": 8,
    "SEP": 9, "OCT": 10, "NOV": 11, "DEC": 12
}

def agreger_par_annee(df):
    resultats = []
    
    for (dept, annee), groupe in df.groupby(["departement", "annee"]):
        row = {"departement": dept, "annee": annee}
        
        for mois_abr, mois_num in mois_map.items():
            ligne_mois = groupe[groupe["MOIS"] == mois_abr]
            if len(ligne_mois) > 0:
                row[f"precip_{mois_num:02d}_mm"]    = ligne_mois["PRECTOTCORR"].values[0]
                row[f"temp_moy_{mois_num:02d}_c"]   = ligne_mois["T2M"].values[0]
                row[f"temp_max_{mois_num:02d}_c"]   = ligne_mois["T2M_MAX"].values[0]
                row[f"temp_min_{mois_num:02d}_c"]   = ligne_mois["T2M_MIN"].values[0]
                row[f"humidite_{mois_num:02d}_pct"] = ligne_mois["RH2M"].values[0]
                row[f"rayonnement_{mois_num:02d}"]  = ligne_mois["ALLSKY_SFC_SW_DWN"].values[0]
        
        row["precip_annuelle_mm"]        = groupe["PRECTOTCORR"].sum()
        row["temp_moy_annuelle_c"]       = groupe["T2M"].mean()
        row["temp_max_annuelle_c"]       = groupe["T2M_MAX"].max()
        row["temp_min_annuelle_c"]       = groupe["T2M_MIN"].min()
        row["humidite_moy_annuelle_pct"] = groupe["RH2M"].mean()
        row["rayonnement_moyen_annuel"]  = groupe["ALLSKY_SFC_SW_DWN"].mean()
        
        etp_estimee = row["rayonnement_moyen_annuel"] * 0.408 * 30 * 12
        row["indice_stress_hydrique"] = (
            row["precip_annuelle_mm"] / etp_estimee
            if etp_estimee > 0 else None
        )
        
        resultats.append(row)
    
    return pd.DataFrame(resultats)

climat_annuel = agreger_par_annee(climat_mensuel)

print(f"Dataset climatique annuel : {climat_annuel.shape}")
print(f"Nombre de colonnes : {len(climat_annuel.columns)}")
print(f"Aperçu des colonnes : {list(climat_annuel.columns)[:15]}...")
climat_annuel.to_csv("../data/processed/climat_annuel.csv", index=False)
print("\nSauvegardé dans data/processed/climat_annuel.csv")

Dataset climatique annuel : (240, 81)
Nombre de colonnes : 81
Aperçu des colonnes : ['departement', 'annee', 'precip_01_mm', 'temp_moy_01_c', 'temp_max_01_c', 'temp_min_01_c', 'humidite_01_pct', 'rayonnement_01', 'precip_02_mm', 'temp_moy_02_c', 'temp_max_02_c', 'temp_min_02_c', 'humidite_02_pct', 'rayonnement_02', 'precip_03_mm']...

Sauvegardé dans data/processed/climat_annuel.csv
